# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Note: dataset.metadata is an object, not a dict
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(dataset.metadata, 'version', 'N/A')}")

## 2. Data Overview
Review available record sets and fields and inspect their `@id`s using `mlcroissant`.

In [ ]:
# Discover record sets
print("Available record sets (by @id):")
for record_set in dataset.metadata.record_sets:
    print(f"- @id: {record_set.id}, name: {record_set.name}")
    print(f"  Description: {getattr(record_set, 'description', '')}")
    # List fields for each record set
    if hasattr(record_set, 'fields'):
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', 'N/A')}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record set ids
record_sets = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set {record_set_id} with shape {dataframes[record_set_id].shape}")
    else:
        print(f"No records found for record set {record_set_id}")

# Example: see the fields for the primary (first) record set
if record_sets:
    first_record_set_id = record_sets[0]
    if first_record_set_id in dataframes:
        print(f"\nColumns in record set {first_record_set_id}:")
        print(dataframes[first_record_set_id].columns.tolist())
        display(dataframes[first_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
# Find a numeric field in the first record set
import numpy as np

record_set_id = record_sets[0] if record_sets else None

numeric_field = None
group_field = None
if record_set_id and record_set_id in dataframes:
    df = dataframes[record_set_id]
    # Try to heuristically find a numeric field
    numeric_candidates = []
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_candidates.append(col)
    if not numeric_candidates:
        # Try to convert columns to numeric if possible
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                pass
        for col in df.columns:
            if np.issubdtype(df[col].dtype, np.number):
                numeric_candidates.append(col)
    print(f"Numeric field candidates: {numeric_candidates}")
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # pick the first
    # Try to select a groupable field (e.g. with a small number of unique categories)
    for col in df.columns:
        if df[col].nunique() > 1 and df[col].nunique() < 10 and col != numeric_field:
            group_field = col
            break

    # Filtering example
    if numeric_field:
        threshold = df[numeric_field].mean()  # mean as reasonable threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field}:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping example
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (mean):")
            display(grouped_df.head())
        else:
            print("No suitable categorical field for grouping found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No records available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we plot the distribution of the numeric field and, if group field is present, compare groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field and record_set_id in dataframes:
    df = dataframes[record_set_id]
    # Plot distribution
    plt.figure(figsize=(7, 5))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {record_set_id}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group if available
    if group_field:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No suitable numeric or group field for visualization.")

## 6. Conclusion
In this notebook, we have loaded and explored a FAIR-compliant Croissant dataset of protein mass spectrometry data using the `mlcroissant` library. We programmatically discovered record sets and fields by `@id`, loaded them into Pandas DataFrames, performed basic filtering and normalization, and visualized numeric distributions.

Key observations:
- The dataset provides complex, structured data as multiple record sets. Field selection by `@id` is required for robust automation.
- Exploratory analysis is easy using Pandas, and visualizations such as histograms and box plots help understand numeric distributions and groupwise comparisons.

Further steps could include advanced analysis specific to the biological context, integration with additional FAIR datasets, or application of machine learning models on the structured result.